# IFC Evacuation Path Analysis — Medical Clinic

This notebook studies evacuation (egress) flow inside `assets/NBU_MedicalClinic_IFC/NBU_MedicalClinic_Arch.ifc` using `topologicpy`, following the recommended workflow:

1. Create a sampled or waypoint graph
2. Make a list of all starting points
3. Make a list of all destination points
4. Compute shortest paths from source to destination
5. Optionally straighten the path (`Wire.Straighten`)
6. Calculate the lengths of the paths (`Wire.Length`)
7. Conduct analysis on worst case, average path, etc.

An agent-based crowd simulation (varying walking abilities, congestion) is **out of scope** for this notebook — it is noted as future work in the final section.

## Note on method

The reference notebook (`assets/References/Graph ML -- NOTEBOOKS/IFC_Path_Analysis.ipynb`) builds its waypoint graph by merging every `IfcSpace` floor face into one `Shell` and grid-slicing it. **That approach fails on this IFC file**: the Revit export gives each room real wall thickness, so adjacent room floor footprints do not share coincident edges — `Shell.ByFaces()` raises `Could not create shell. Returning None.`, and even a `Cluster`-based workaround produces a navigation graph too sparse to route across (`Graph.ShortestPath` returns `None` between rooms).

Instead this notebook uses a **room-centroid graph bridged through real door openings** — the same idea as the course's earlier OBJ-based notebook (`GraphML_RaniaChihaoui.ipynb`), built here from the IFC's own door/space adjacency (`topologicpy.IFC.AccessGraph`). This routes people through actual doorways, never through walls, which is what evacuation analysis needs. Section 3 below still demonstrates the literal grid/waypoint-sampling method from the reference notebook on a single room, to show the technique and why it doesn't scale to the whole floor here.

In [1]:
import time
import re

from topologicpy.IFC import IFC
from topologicpy.Topology import Topology
from topologicpy.Dictionary import Dictionary
from topologicpy.Face import Face
from topologicpy.Wire import Wire
from topologicpy.Vertex import Vertex
from topologicpy.Cluster import Cluster
from topologicpy.Grid import Grid
from topologicpy.Graph import Graph

IFC_PATH = "assets/NBU_MedicalClinic_IFC/NBU_MedicalClinic_Arch.ifc"
t0 = time.time()

## 1. Load the IFC file and extract spaces & doors

`topologicpy.IFC` has its own fast IFC parser and does not need `ifcopenshell`.

In [2]:
f = IFC.FileByPath(IFC_PATH)

spaces = IFC.TopologiesByPath(IFC_PATH, includeTypes=["ifcspace"], dictionaryMode="all", silent=True)
doors = IFC.TopologiesByPath(IFC_PATH, includeTypes=["ifcdoor"], dictionaryMode="all", silent=True)
print(f"[{time.time()-t0:.1f}s] spaces={len(spaces)} doors={len(doors)}")

space_by_id = {}
for s in spaces:
    d = Topology.Dictionary(s)
    sid = Dictionary.ValueAtKey(d, "IFC_id")
    space_by_id[sid] = {
        "topology": s,
        "centroid": Topology.Centroid(s),
        "name": str(Dictionary.ValueAtKey(d, "IFC_name") or ""),
        "long_name": str(Dictionary.ValueAtKey(d, "IFC_long_name") or ""),
        "level": Dictionary.ValueAtKey(d, "IFC_pset_PSet_Revit_Constraints_Level"),
        "area": Dictionary.ValueAtKey(d, "IFC_qto_GSA_Space_Areas_GSA_BIM_Area"),
    }

door_by_id = {}
for dr in doors:
    d = Topology.Dictionary(dr)
    did = Dictionary.ValueAtKey(d, "IFC_id")
    door_by_id[did] = {"topology": dr, "centroid": Topology.Centroid(dr)}

levels = sorted({info["level"] for info in space_by_id.values() if info["level"]})
print("Levels:", levels)
for lvl in levels:
    n = sum(1 for info in space_by_id.values() if info["level"] == lvl)
    print(f"  {lvl}: {n} spaces")

IFCFastTopology.Parse - Parsed 273137 entities in 3.045s.


[42.4s] spaces=266 doors=254


Levels: ['First Floor', 'Roof - Main', 'Second Floor']
  First Floor: 154 spaces
  Roof - Main: 6 spaces
  Second Floor: 106 spaces


## 2. Identify exits and vertical circulation

`IFC.DoorSpaceCardinalityReport` tells us, for every door, how many `IfcSpace` entities it borders:
- `space_count == 1` and `connector_type == "IfcDoor"` → a **true exterior exit** (the door touches only one interior space, so the other side is outside).
- `space_count == 2` → a normal interior door connecting two rooms.

Stairwell cores are identified by `"STAIR"` appearing in the space name/long name, then paired First Floor ↔ Second Floor by their shared letter code (e.g. `1CS3` ↔ `2CS3`).

In [3]:
report = IFC.DoorSpaceCardinalityReport(f, silent=True)
exterior = [r for r in report if r.get("connector_type") == "IfcDoor" and r.get("space_count") == 1]
exit_room_ids = {r["space_ids"][0] for r in exterior if r.get("space_ids")}

print(f"Exterior exit doors: {len(exterior)} -> exit rooms: {len(exit_room_ids)}")
for sid in exit_room_ids:
    info = space_by_id.get(sid, {})
    print("  exit room:", info.get("name"), "|", info.get("long_name"), "|", info.get("level"))

stair_spaces = {
    sid: info for sid, info in space_by_id.items()
    if "STAIR" in info["name"].upper() or "STAIR" in info["long_name"].upper()
}

def stair_key(name):
    return re.sub(r"^\d+", "", name)

ff_stairs = {stair_key(info["name"]): sid for sid, info in stair_spaces.items() if info["level"] == "First Floor"}
sf_stairs = {stair_key(info["name"]): sid for sid, info in stair_spaces.items() if info["level"] == "Second Floor"}
stair_pairs = [(ff_stairs[k], sf_stairs[k]) for k in ff_stairs if k in sf_stairs]

print(f"\nStairwell spaces found: {len(stair_spaces)}")
for sid, info in stair_spaces.items():
    print("  ", info["name"], "|", info["long_name"], "|", info["level"])
print(f"\nMatched First<->Second floor stair pairs: {len(stair_pairs)}")
for ff_id, sf_id in stair_pairs:
    print("  ", space_by_id[ff_id]["name"], "<->", space_by_id[sf_id]["name"])

Exterior exit doors: 10 -> exit rooms: 10
  exit room: 1EC4 | CORRIDOR | First Floor
  exit room: 1E15-A | GAS STORAGE | First Floor
  exit room: 1e+17 | HAZ. STOR. | First Floor
  exit room: 1EC1 | CORRIDOR | First Floor
  exit room: 1e+20 | TRASH | First Floor
  exit room: 1CC3 | CORRIDOR | First Floor
  exit room: 1AC3 | CORRIDOR | First Floor
  exit room: 1e+21 | FLAMABLE STORAGE | First Floor
  exit room: 1000000000000000.0 | CENTRAL MECH. RM. | First Floor
  exit room: 1AC5 | CORRIDOR | First Floor

Stairwell spaces found: 6
   1CS3 | STAIR | First Floor
   1BS2 | STAIR | First Floor
   1AS1 | STAIR | First Floor
   2AS1 | STAIR | Second Floor
   2BS2 | STAIR | Second Floor
   2CS3 | STAIR | Second Floor

Matched First<->Second floor stair pairs: 3
   1CS3 <-> 2CS3
   1BS2 <-> 2BS2
   1AS1 <-> 2AS1


## 3. Step 1 (literal demo) — Fine grid/waypoint sampling on a single room

This reproduces the reference notebook's exact method — `Wire.BoundingRectangle` → `Grid.EdgesByDistances` → `Topology.Slice` → `Graph.ByTopology` — on the **largest First Floor room**, to demonstrate the technique on geometry where it actually works (a single watertight footprint).

In [4]:
def get_floor_face(space_topology):
    """Pick the downward-facing (normal z < -0.9) face with the lowest centroid Z as the room's floor footprint.
    Real-world Revit-exported IfcSpace geometry is a Cluster of triangulated faces, not a solid Cell,
    so Cell.Decompose() cannot be used here."""
    floor_faces = []
    for fa in Topology.Faces(space_topology):
        try:
            if Face.Normal(fa)[2] < -0.9:
                floor_faces.append(fa)
        except Exception:
            pass
    if not floor_faces:
        return None
    floor_faces.sort(key=lambda ff: Topology.Centroid(ff).Z())
    return floor_faces[0]

ff_ids = [sid for sid, info in space_by_id.items() if info["level"] == "First Floor"]
largest_id = max(ff_ids, key=lambda sid: (space_by_id[sid]["area"] or 0))
print("Largest First Floor room:", space_by_id[largest_id]["name"], f"({space_by_id[largest_id]['area']:.1f} sqm)")

room_fp = get_floor_face(space_by_id[largest_id]["topology"])
br = Wire.BoundingRectangle(room_fp)
bd = Topology.Dictionary(br)
width = Dictionary.ValueAtKey(bd, "width")
length = Dictionary.ValueAtKey(bd, "length")

grid_spacing = 1.0
uRange = list(range(0, int(width) + int(grid_spacing) + 1, int(grid_spacing)))
vRange = list(range(0, int(length) + int(grid_spacing) + 1, int(grid_spacing)))
grid = Grid.EdgesByDistances(room_fp, clip=True, uRange=uRange, vRange=vRange)
sliced = Topology.Slice(room_fp, grid)
nav_demo = Graph.ByTopology(sliced, direct=False, viaSharedTopologies=True)
print(f"Waypoint graph: {len(Graph.Vertices(nav_demo))} vertices, {len(Graph.Edges(nav_demo))} edges")

gv = Graph.Vertices(nav_demo)
demo_path = Graph.ShortestPath(nav_demo, vertexA=gv[0], vertexB=gv[-1])
print("Sample within-room path found:", demo_path is not None, "| length:", Wire.Length(demo_path) if demo_path else None)

Largest First Floor room: 1DC1 (182.3 sqm)


Waypoint graph: 593 vertices, 764 edges
Sample within-room path found: True | length: 25.894517


### Why this doesn't scale to the whole floor

Merging all 154 First Floor room footprints into one `Shell` and repeating the same grid-slice fails:

```
Shell.ByFaces - Error: Could not create shell. Returning None.
```

Even after working around this by using a `Cluster` instead of a `Shell`, the resulting grid-sliced navigation graph is fragmented (only ~120 edges across ~230 vertices for 154 rooms), and `Graph.ShortestPath` between rooms in different parts of the building returns `None`. The cause: this Revit export gives walls real thickness, so neighboring room footprints never share a coincident edge for the grid to connect across — geometrically they are gaps, not boundaries. Routing through a building like this has to go through the actual modeled door openings, which Section 4 does.

## 4. Step 1 (building-scale) — Waypoint graph bridged through real doors

`IFC.AccessGraph` builds a graph of `IfcSpace` vertices (positioned at room centroids, `importMode="geometry"`) connected by edges wherever a door (or other connector) joins two spaces — exactly the door-bridging idea from the course's own residential OBJ notebook, but derived directly from the IFC model's door/space relationships.

In [5]:
g = IFC.AccessGraph(f, importMode="geometry", dictionaryMode="all", silent=True)

vertex_by_ifc_id = {}
for v in Graph.Vertices(g):
    d = Topology.Dictionary(v)
    vid = Dictionary.ValueAtKey(d, "IFC_id")
    if vid is not None:
        vertex_by_ifc_id[vid] = v

print(f"Access graph: {len(Graph.Vertices(g))} vertices, {len(Graph.Edges(g))} edges")

components = Graph.ConnectedComponents(g)
vertex_component = {}
for i, comp in enumerate(components):
    for v in Graph.Vertices(comp):
        d = Topology.Dictionary(v)
        vertex_component[Dictionary.ValueAtKey(d, "IFC_id")] = i
comp_sizes = [len(Graph.Vertices(c)) for c in components]
print(f"Connected components: {len(components)} | sizes (top 10): {sorted(comp_sizes, reverse=True)[:10]}")

no_door = [sid for sid, info in space_by_id.items() if vertex_component.get(sid) is None]
isolated = [sid for sid in space_by_id if vertex_component.get(sid) is not None
            and comp_sizes[vertex_component[sid]] == 1 and sid not in exit_room_ids]
print(f"\nRooms with no modeled door connection at all (no graph vertex): {len(no_door)}")
print(f"Single-room components with no exit and no neighbor (flagged, no egress path): {len(isolated)}")
for sid in isolated[:15]:
    print("  -", space_by_id[sid]["name"], "|", space_by_id[sid]["long_name"], "|", space_by_id[sid]["level"])

Access graph: 269 vertices, 232 edges


Connected components: 48 | sizes (top 10): [85, 52, 19, 19, 11, 9, 7, 7, 6, 5]

Rooms with no modeled door connection at all (no graph vertex): 0
Single-room components with no exit and no neighbor (flagged, no egress path): 26
  - 1A01 | PATIENT ADMIN. RECEPT. | First Floor
  - 1A02 | RMO ANALYST | First Floor
  - 1A14 | SUPER / NCOIC | First Floor
  - 1C01 | WAITING | First Floor
  - 1D03 | CLEAN U. | First Floor
  - 1D04 | TOILET | First Floor
  - 1D15 | WTS & MEAS. ROOM | First Floor
  - 1D37 | CLEAN U. | First Floor
  - E1 | ELEVATOR | First Floor
  - 2AS1 | STAIR | Second Floor
  - 2A05 | OH DTR #1 | Second Floor
  - 2A06 | OH DTR #2 | Second Floor
  - 2A07 | OH DTR #3 | Second Floor
  - 2A08 | GEN. DENT. DTR #4 | Second Floor
  - 2A09 | GEN. DENT. DTR #5 | Second Floor


This fragmentation is a real property of this IFC model, not a computation bug — about half the spaces are not reachable from each other through modeled door connectors alone. Some are legitimately self-contained (e.g. `GAS STORAGE`, `TRASH`, `FLAMABLE STORAGE` have their own direct exterior door and no interior connection by design — these are *correctly* isolated). Others (e.g. open-plan reception/waiting areas) likely have no door object modeled in the IFC at all. The analysis below only routes between rooms in the **same connected component**, and explicitly reports rooms with no modeled egress path as a finding rather than silently dropping them.

## 5. Steps 2–3 — Sources and destinations

In [6]:
ff_room_ids = [sid for sid, info in space_by_id.items() if info["level"] == "First Floor"]
sf_room_ids = [sid for sid, info in space_by_id.items() if info["level"] == "Second Floor"]

print(f"Sources (Step 2): {len(ff_room_ids)} First Floor rooms, {len(sf_room_ids)} Second Floor rooms")
print(f"Destinations (Step 3): {len(exit_room_ids)} First Floor exterior-exit rooms, {len(stair_pairs)} stairwell pairs")

Sources (Step 2): 154 First Floor rooms, 106 Second Floor rooms
Destinations (Step 3): 10 First Floor exterior-exit rooms, 3 stairwell pairs


## 6. Steps 4–6 — Shortest paths, straighten, lengths (First Floor)

For each First Floor room, find the nearest exit room (smallest `Wire.Length`), restricted to rooms in the same connected component. `Wire.Straighten` is applied for completeness; since graph edges here are already straight room-centroid-to-room-centroid segments (no fine grid staircase pattern to smooth), the straightened length is expected to equal the raw length — unlike the Section 3 fine-grid demo, where straightening does shorten the path.

In [7]:
ff_footprints = [fp for sid in ff_room_ids for fp in [get_floor_face(space_by_id[sid]["topology"])] if fp]
ff_floor_plan = Cluster.ByTopologies(ff_footprints)
print(f"[{time.time()-t0:.1f}s] First Floor footprints extracted: {len(ff_footprints)}/{len(ff_room_ids)}")

[59.3s] First Floor footprints extracted: 154/154


In [8]:
def nearest_destination(room_id, candidate_ids, graph, vertex_by_id, vertex_component):
    """Shortest path (Step 4) from room_id to the nearest reachable candidate, restricted to the same connected component."""
    vA = vertex_by_id.get(room_id)
    comp_a = vertex_component.get(room_id)
    if vA is None or comp_a is None:
        return None
    best = None
    for cid in candidate_ids:
        if cid == room_id:
            return (cid, 0.0, None)
        if vertex_component.get(cid) != comp_a:
            continue
        vB = vertex_by_id.get(cid)
        if vB is None:
            continue
        path = Graph.ShortestPath(graph, vertexA=vA, vertexB=vB)
        if path is None:
            continue
        length = Wire.Length(path)
        if best is None or length < best[1]:
            best = (cid, length, path)
    return best

ff_results = {}
ff_no_path = []
t1 = time.time()
for rid in ff_room_ids:
    res = nearest_destination(rid, exit_room_ids, g, vertex_by_ifc_id, vertex_component)
    if res is None:
        ff_no_path.append(rid)
        continue
    exit_id, raw_length, path = res
    if path is not None:
        straight_path = Wire.Straighten(path, host=ff_floor_plan)
        straight_length = Wire.Length(straight_path)
    else:
        straight_path, straight_length = None, 0.0
    ff_results[rid] = {
        "exit_id": exit_id, "raw_length": raw_length, "straight_length": straight_length,
        "path": path, "straight_path": straight_path,
    }

print(f"[{time.time()-t1:.1f}s] First Floor: nearest-exit computed for {len(ff_results)}/{len(ff_room_ids)} rooms")
print(f"Rooms with no path to any exit (flagged): {len(ff_no_path)}")
for rid in ff_no_path[:15]:
    print("  -", space_by_id[rid]["name"], "|", space_by_id[rid]["long_name"])

Wire.IsManifold - Error: The input wire parameter is not a valid topologic wire. Returning None.


caller name: RemoveCollinearEdges
Wire.IsManifold - Error: The input wire parameter is not a valid topologic wire. Returning None.
caller name: RemoveCollinearEdges
Wire.IsManifold - Error: The input wire parameter is not a valid topologic wire. Returning None.
caller name: RemoveCollinearEdges


Wire.IsManifold - Error: The input wire parameter is not a valid topologic wire. Returning None.
caller name: RemoveCollinearEdges
Wire.IsManifold - Error: The input wire parameter is not a valid topologic wire. Returning None.
caller name: RemoveCollinearEdges


Wire.IsManifold - Error: The input wire parameter is not a valid topologic wire. Returning None.
caller name: RemoveCollinearEdges
Wire.IsManifold - Error: The input wire parameter is not a valid topologic wire. Returning None.
caller name: RemoveCollinearEdges


Wire.IsManifold - Error: The input wire parameter is not a valid topologic wire. Returning None.
caller name: RemoveCollinearEdges


...

Wire.IsManifold - Error: The input wire parameter is not a valid topologic wire. Returning None.
caller name: RemoveCollinearEdges
Wire.IsManifold - Error: The input wire parameter is not a valid topologic wire. Returning None.
caller name: RemoveCollinearEdges
Wire.IsManifold - Error: The input wire parameter is not a valid topologic wire. Returning None.


caller name: RemoveCollinearEdges
Wire.IsManifold - Error: The input wire parameter is not a valid topologic wire. Returning None.
caller name: RemoveCollinearEdges


Wire.IsManifold - Error: The input wire parameter is not a valid topologic wire. Returning None.
caller name: RemoveCollinearEdges
Wire.IsManifold - Error: The input wire parameter is not a valid topologic wire. Returning None.
caller name: RemoveCollinearEdges
Wire.IsManifold - Error: The input wire parameter is not a valid topologic wire. Returning None.
caller name: RemoveCollinearEdges


Wire.IsManifold - Error: The input wire parameter is not a valid topologic wire. Returning None.
caller name: RemoveCollinearEdges


[55.2s] First Floor: nearest-exit computed for 111/154 rooms
Rooms with no path to any exit (flagged): 43
  - 1A01 | PATIENT ADMIN. RECEPT.
  - 1A02 | RMO ANALYST
  - 1A14 | SUPER / NCOIC
  - 1DC3 | CORRIDOR
  - 1CC2 | CORRIDOR
  - 1C01 | WAITING
  - 1C05 | VISUAL FIELD
  - 1C06 | SOIL. UTL. / TRASH
  - 1C08 | SCREEN EYE TEST
  - 1C09 | FUNDUS CAMERA ROOM
  - 1D03 | CLEAN U.
  - 1D04 | TOILET
  - 1D05 | TRMT RM. DIRTY PROC.
  - 1D06 | SCOPE WASH / DECON.
  - 1D07 | INTERACTION STATION


## 7. Multi-floor combined analysis — Second Floor via stairwells

Second Floor has **no** exterior exit doors of its own — all 10 are on First Floor. Each Second Floor room routes to its nearest stairwell core; total egress distance is:

`distance(room → stair on Second Floor) + stair_travel_distance + distance(matched stair on First Floor → nearest exit)`

`stair_travel_distance` approximates the walked distance along the stair flights/landings as `2 x floor-to-floor elevation` (4.57 m × 2 ≈ 9.14 m) — a simplifying assumption noted here since this notebook does not trace the actual `IfcStair` flight geometry.

In [9]:
STAIR_TRAVEL_DISTANCE = 4.57 * 2.0  # approximation: 2x floor-to-floor elevation along stair flights/landings

stair_sf_to_ff = {sf_id: ff_id for ff_id, sf_id in stair_pairs}

sf_results = {}
sf_no_path = []
t2 = time.time()
for rid in sf_room_ids:
    res = nearest_destination(rid, set(stair_sf_to_ff.keys()), g, vertex_by_ifc_id, vertex_component)
    if res is None:
        sf_no_path.append(rid)
        continue
    stair_sf_id, dist_to_stair, path = res
    stair_ff_id = stair_sf_to_ff[stair_sf_id]
    ff_exit = ff_results.get(stair_ff_id)
    if ff_exit is None:
        sf_no_path.append(rid)
        continue
    total = dist_to_stair + STAIR_TRAVEL_DISTANCE + ff_exit["raw_length"]
    sf_results[rid] = {
        "stair_sf_id": stair_sf_id, "stair_ff_id": stair_ff_id,
        "dist_to_stair": dist_to_stair, "stair_travel": STAIR_TRAVEL_DISTANCE,
        "ff_exit_leg": ff_exit["raw_length"], "total_length": total,
        "path": path,
    }

print(f"[{time.time()-t2:.1f}s] Second Floor: combined egress computed for {len(sf_results)}/{len(sf_room_ids)} rooms")
print(f"Rooms with no path to any stairwell (flagged): {len(sf_no_path)}")
for rid in sf_no_path[:15]:
    print("  -", space_by_id[rid]["name"], "|", space_by_id[rid]["long_name"])

[2.6s] Second Floor: combined egress computed for 72/106 rooms
Rooms with no path to any stairwell (flagged): 34
  - 2AC2 | CORRIDOR
  - 2AC3 | CORRIDOR
  - 2AC4 | CORRIDOR
  - 2AC5 | CORRIDOR
  - 2A01 | LIBRARY / CONF. ROOM
  - 2A02 | SUPER OFFICE
  - 2A03 | SEC. OFFICE
  - 2A04 | DENTAL SURGEON OFFICE
  - 2A05 | OH DTR #1
  - 2A06 | OH DTR #2
  - 2A07 | OH DTR #3
  - 2A08 | GEN. DENT. DTR #4
  - 2A09 | GEN. DENT. DTR #5
  - 2A10 | DENTAL WAITING
  - 2A11 | DENTAL WAITING


## 8. Step 7 — Worst case / average / best case analysis

In [10]:
def summarize(label, lengths_by_room):
    if not lengths_by_room:
        print(f"{label}: no data")
        return None
    worst_id = max(lengths_by_room, key=lambda k: lengths_by_room[k])
    best_id = min(lengths_by_room, key=lambda k: lengths_by_room[k])
    avg = sum(lengths_by_room.values()) / len(lengths_by_room)
    print(f"--- {label} ({len(lengths_by_room)} rooms) ---")
    print(f"  Worst case: {space_by_id[worst_id]['name']:<10s} {lengths_by_room[worst_id]:6.1f} m")
    print(f"  Best case:  {space_by_id[best_id]['name']:<10s} {lengths_by_room[best_id]:6.1f} m")
    print(f"  Average:    {avg:6.1f} m")
    return worst_id, best_id, avg

ff_lengths = {rid: r["raw_length"] for rid, r in ff_results.items()}
sf_lengths = {rid: r["total_length"] for rid, r in sf_results.items()}
combined_lengths = {**ff_lengths, **sf_lengths}

ff_stats = summarize("First Floor (direct to exterior exit)", ff_lengths)
sf_stats = summarize("Second Floor (via stairwell, combined)", sf_lengths)
combined_stats = summarize("Whole building combined", combined_lengths)

print(f"\nTotal rooms with NO modeled egress path: {len(ff_no_path) + len(sf_no_path)} "
      f"({len(ff_no_path)} First Floor, {len(sf_no_path)} Second Floor)")

--- First Floor (direct to exterior exit) (111 rooms) ---
  Worst case: 1D38         49.4 m
  Best case:  1AC3          0.0 m
  Average:      20.4 m
--- Second Floor (via stairwell, combined) (72 rooms) ---
  Worst case: 2C18         89.1 m
  Best case:  2CS3         11.4 m
  Average:      54.3 m
--- Whole building combined (183 rooms) ---
  Worst case: 2C18         89.1 m
  Best case:  1AC3          0.0 m
  Average:      33.7 m

Total rooms with NO modeled egress path: 77 (43 First Floor, 34 Second Floor)


In [11]:
rows = []
for rid, r in ff_results.items():
    rows.append((space_by_id[rid]["name"], "First Floor", space_by_id[r["exit_id"]]["name"], r["raw_length"], r["straight_length"]))
for rid, r in sf_results.items():
    rows.append((space_by_id[rid]["name"], "Second Floor", space_by_id[r["stair_ff_id"]]["name"] + " (via stair)", r["total_length"], r["total_length"]))

rows.sort(key=lambda row: row[3], reverse=True)
print(f"{'Room':<12s} {'Floor':<13s} {'Nearest exit/stair':<22s} {'Raw (m)':>8s} {'Straight (m)':>13s}")
for row in rows[:25]:
    print(f"{row[0]:<12s} {row[1]:<13s} {row[2]:<22s} {row[3]:8.1f} {row[4]:13.1f}")

Room         Floor         Nearest exit/stair      Raw (m)  Straight (m)
2C18         Second Floor  1BS2 (via stair)           89.1          89.1
2C22         Second Floor  1BS2 (via stair)           88.8          88.8
2C19         Second Floor  1BS2 (via stair)           88.2          88.2
2C08         Second Floor  1BS2 (via stair)           86.2          86.2
2CC1         Second Floor  1BS2 (via stair)           83.7          83.7
2R02         Second Floor  1BS2 (via stair)           83.7          83.7
2D05         Second Floor  1BS2 (via stair)           78.9          78.9
2D11         Second Floor  1BS2 (via stair)           74.8          74.8
2D09         Second Floor  1BS2 (via stair)           74.6          74.6
2B5          Second Floor  1BS2 (via stair)           71.9          71.9
2B25         Second Floor  1BS2 (via stair)           71.0          71.0
2B23         Second Floor  1BS2 (via stair)           70.9          70.9
2B24         Second Floor  1BS2 (via stair)        

## 9. Visualization

Two views, one per floor:

- **First Floor:** every room's egress path is drawn (not just the worst case), color-coded **green → yellow → red** from shortest to longest. The worst-case room's path is thicker and labeled.
- **Second Floor:** the room → nearest-stairwell leg only (the segment this notebook actually computes a `Wire` for), colored by each room's **total combined egress length** (including stair travel + the First Floor leg to the real exit) — so a short line on screen can still be colored red if that room is far from an exit overall.

Note: every path is a straight room-centroid-to-room-centroid segment (see Section 4) — it does **not** bend through corridors like a real walked route, it represents the door-to-door graph connectivity and its total length.

## 9a. Building overview — floor plan with exits & stairwells

Before looking at egress paths, here is the spatial layout of the building:
- **Gray** = all rooms (First Floor left, Second Floor right)
- **Green squares** = exterior exit doors (rooms with direct access to outside)
- **Blue triangles** = stairwell cores connecting the two floors
- **Orange** = rooms with no modelled egress path (flagged for review)

Hover any room to see its name, long name, floor, and area.

In [ ]:
import plotly.graph_objects as go

# helper functions reused by all three visualization cells
def face_outline_xy(face):
    wires = Topology.Wires(face)
    verts = Topology.Vertices(wires[0] if wires else face)
    xs = [v.X() for v in verts] + [verts[0].X()]
    ys = [v.Y() for v in verts] + [verts[0].Y()]
    return xs, ys

def wire_xy(wire):
    verts = Topology.Vertices(wire)
    return [v.X() for v in verts], [v.Y() for v in verts]

def len_to_color(length, mn, mx):
    t = 0.0 if mx <= mn else max(0.0, min(1.0, (length - mn) / (mx - mn)))
    if t < 0.5:
        return f"rgb({int(510*t)},200,0)"
    return f"rgb(255,{int(200*(1-(t-0.5)*2))},0)"

# ── building overview ──────────────────────────────────────────────────────────
no_path_ids = set(ff_no_path + sf_no_path)
stair_sf_ids = {sf_id for _, sf_id in stair_pairs}
stair_ff_ids = {ff_id for ff_id, _ in stair_pairs}
all_stair_ids = stair_sf_ids | stair_ff_ids

def room_fill_color(sid):
    if sid in exit_room_ids:  return 'rgba(34,139,34,0.55)'
    if sid in all_stair_ids:  return 'rgba(30,100,200,0.45)'
    if sid in no_path_ids:    return 'rgba(230,120,20,0.35)'
    return 'rgba(200,200,205,0.25)'

def room_line_color(sid):
    if sid in exit_room_ids:  return 'rgba(0,100,0,0.9)'
    if sid in all_stair_ids:  return 'rgba(10,40,160,0.9)'
    if sid in no_path_ids:    return 'rgba(160,70,0,0.8)'
    return 'rgba(100,100,110,0.5)'

fig0 = go.Figure()

all_room_ids = [sid for sid, info in space_by_id.items()
                if info["level"] in ("First Floor", "Second Floor")]

# ── room footprints ────────────────────────────────────────────────────────────
for sid in all_room_ids:
    info = space_by_id[sid]
    fp = get_floor_face(info["topology"])
    if fp is None:
        continue
    xs, ys = face_outline_xy(fp)
    area = info["area"]
    area_str = f"{area:.1f} sqm" if area else "n/a"
    fig0.add_trace(go.Scatter(
        x=xs, y=ys, fill='toself',
        fillcolor=room_fill_color(sid),
        line=dict(color=room_line_color(sid), width=0.8),
        mode='lines', showlegend=False,
        hovertemplate=(
            f"<b>{info['name']}</b><br>{info['long_name']}<br>"
            f"{info['level']}<br>{area_str}<extra></extra>"
        ),
    ))

# ── graph edges (all, thin gray) ───────────────────────────────────────────────
edge_xs, edge_ys = [], []
for e in Graph.Edges(g):
    everts = Topology.Vertices(e)
    if len(everts) == 2:
        edge_xs += [everts[0].X(), everts[1].X(), None]
        edge_ys += [everts[0].Y(), everts[1].Y(), None]
fig0.add_trace(go.Scatter(
    x=edge_xs, y=edge_ys, mode='lines',
    line=dict(color='rgba(80,80,100,0.35)', width=1.2),
    showlegend=False, hoverinfo='skip',
))

# ── graph nodes (all room centroids in graph) ──────────────────────────────────
node_xs = [space_by_id[sid]["centroid"].X() for sid in all_room_ids if sid in vertex_by_ifc_id]
node_ys = [space_by_id[sid]["centroid"].Y() for sid in all_room_ids if sid in vertex_by_ifc_id]
fig0.add_trace(go.Scatter(
    x=node_xs, y=node_ys, mode='markers',
    marker=dict(color='white', size=7, symbol='circle',
                line=dict(color='rgba(60,60,80,0.8)', width=1.5)),
    showlegend=False, hoverinfo='skip',
))

for sid in exit_room_ids:
    v = space_by_id[sid]["centroid"]
    fig0.add_trace(go.Scatter(
        x=[v.X()], y=[v.Y()], mode='markers+text',
        marker=dict(color='green', size=13, symbol='square',
                    line=dict(color='darkgreen', width=1.5)),
        text=[space_by_id[sid]["name"]],
        textposition='top center', textfont=dict(size=8, color='darkgreen'),
        showlegend=False,
        hovertemplate=f"EXIT: {space_by_id[sid]['name']}<br>{space_by_id[sid]['long_name']}<extra></extra>",
    ))

for ff_id, sf_id in stair_pairs:
    for sid in (ff_id, sf_id):
        v = space_by_id[sid]["centroid"]
        lvl = "FF" if space_by_id[sid]["level"] == "First Floor" else "SF"
        fig0.add_trace(go.Scatter(
            x=[v.X()], y=[v.Y()], mode='markers+text',
            marker=dict(color='steelblue', size=13, symbol='triangle-up',
                        line=dict(color='navy', width=1.5)),
            text=[f"{space_by_id[sid]['name']} ({lvl})"],
            textposition='top center', textfont=dict(size=8, color='navy'),
            showlegend=False,
            hovertemplate=f"STAIR: {space_by_id[sid]['name']} — {space_by_id[sid]['level']}<extra></extra>",
        ))

fig0.add_annotation(
    x=0.01, y=0.99, xref="paper", yref="paper",
    align="left", showarrow=False,
    bgcolor="white", bordercolor="#ccc", borderwidth=1,
    text=(
        "<b>Legend</b><br>"
        "<span style='color:green'>■</span> Exit room<br>"
        "<span style='color:steelblue'>▲</span> Stairwell<br>"
        "<span style='color:darkorange'>■</span> No egress path<br>"
        "<span style='color:#888'>■</span> Regular room"
    ),
    font=dict(size=11),
)

fig0.update_layout(
    title=dict(
        text="Building overview — exits, stairwells and flagged rooms<br>"
             "<sup>First Floor and Second Floor shown together · hover any room for details</sup>",
        font=dict(size=15), x=0.5,
    ),
    width=1100, height=800,
    xaxis=dict(scaleanchor="y", scaleratio=1, showgrid=False, zeroline=False, title="X (m)"),
    yaxis=dict(showgrid=False, zeroline=False, title="Y (m)"),
    paper_bgcolor="white", plot_bgcolor="#f5f5f7",
    margin=dict(l=40, r=40, t=90, b=40),
)
fig0.show()

In [ ]:
ff_path_lengths = [r["raw_length"] for r in ff_results.values() if r["path"] is not None]
mn, mx = min(ff_path_lengths), max(ff_path_lengths)
worst_ff_id = ff_stats[0] if ff_stats else None

fig = go.Figure()

# ── room footprints ────────────────────────────────────────────────────────────
for fp in ff_footprints:
    xs, ys = face_outline_xy(fp)
    fig.add_trace(go.Scatter(
        x=xs, y=ys, fill='toself',
        fillcolor='rgba(200,200,205,0.15)',
        line=dict(color='rgba(120,120,130,0.4)', width=0.4),
        mode='lines', showlegend=False, hoverinfo='skip',
    ))

# ── FF graph edges (thin gray) ─────────────────────────────────────────────────
ff_id_set = set(ff_room_ids)
ff_edge_xs, ff_edge_ys = [], []
for e in Graph.Edges(g):
    everts = Topology.Vertices(e)
    if len(everts) == 2:
        d0 = Topology.Dictionary(everts[0])
        d1 = Topology.Dictionary(everts[1])
        id0 = Dictionary.ValueAtKey(d0, "IFC_id")
        id1 = Dictionary.ValueAtKey(d1, "IFC_id")
        if id0 in ff_id_set and id1 in ff_id_set:
            ff_edge_xs += [everts[0].X(), everts[1].X(), None]
            ff_edge_ys += [everts[0].Y(), everts[1].Y(), None]
fig.add_trace(go.Scatter(
    x=ff_edge_xs, y=ff_edge_ys, mode='lines',
    line=dict(color='rgba(100,100,120,0.3)', width=1.2),
    showlegend=False, hoverinfo='skip',
))

# ── FF graph nodes (all room centroids) ────────────────────────────────────────
ff_node_xs = [space_by_id[sid]["centroid"].X() for sid in ff_room_ids if sid in vertex_by_ifc_id]
ff_node_ys = [space_by_id[sid]["centroid"].Y() for sid in ff_room_ids if sid in vertex_by_ifc_id]
fig.add_trace(go.Scatter(
    x=ff_node_xs, y=ff_node_ys, mode='markers',
    marker=dict(color='white', size=7, symbol='circle',
                line=dict(color='rgba(80,80,100,0.6)', width=1.5)),
    showlegend=False, hoverinfo='skip',
))

# ── egress paths (colored lines + nodes) ──────────────────────────────────────
for rid, r in ff_results.items():
    if r["path"] is None:
        continue
    xs, ys = wire_xy(r["path"])
    is_worst = rid == worst_ff_id
    color = len_to_color(r["raw_length"], mn, mx)
    fig.add_trace(go.Scatter(
        x=xs, y=ys, mode='lines+markers',
        line=dict(color=color, width=6 if is_worst else 3),
        marker=dict(color='white', size=10 if is_worst else 7, symbol='circle',
                    line=dict(color=color, width=2.5 if is_worst else 1.8)),
        showlegend=False,
        hovertemplate=(
            f"{space_by_id[rid]['name']} | {space_by_id[rid]['long_name']}<br>"
            f"→ {space_by_id[r['exit_id']]['name']}<br>"
            f"{r['raw_length']:.1f} m<extra></extra>"
        ),
    ))

fig.add_trace(go.Scatter(
    x=[None], y=[None], mode='markers',
    marker=dict(
        colorscale=[[0,'rgb(0,200,0)'],[0.5,'rgb(255,200,0)'],[1,'rgb(255,0,0)']],
        cmin=round(mn,1), cmax=round(mx,1), color=[mn, mx],
        colorbar=dict(title=dict(text="Distance (m)", side="right"),
                      thickness=14, len=0.55, x=1.02, y=0.5, tickfont=dict(size=11)),
        showscale=True,
    ), showlegend=False,
))

for sid in exit_room_ids:
    v = space_by_id[sid]["centroid"]
    fig.add_trace(go.Scatter(
        x=[v.X()], y=[v.Y()], mode='markers+text',
        marker=dict(color='green', size=14, symbol='square', line=dict(color='darkgreen', width=1.5)),
        text=[space_by_id[sid]["name"]], textposition='top center',
        textfont=dict(size=9, color='darkgreen'), showlegend=False,
        hovertemplate=f"EXIT: {space_by_id[sid]['name']}<br>{space_by_id[sid]['long_name']}<extra></extra>",
    ))

if worst_ff_id:
    v = space_by_id[worst_ff_id]["centroid"]
    fig.add_trace(go.Scatter(
        x=[v.X()], y=[v.Y()], mode='markers+text',
        marker=dict(color='darkred', size=18, symbol='star'),
        text=[f"WORST: {space_by_id[worst_ff_id]['name']} ({ff_results[worst_ff_id]['raw_length']:.0f} m)"],
        textposition='bottom center', textfont=dict(size=11, color='darkred'), showlegend=False,
    ))

fig.update_layout(
    title=dict(
        text="First Floor — Egress distance to nearest exit<br>"
             "<sup>Green = short · Red = long · Star = worst case · Square = exit</sup>",
        font=dict(size=15), x=0.5,
    ),
    width=1100, height=750,
    xaxis=dict(scaleanchor="y", scaleratio=1, showgrid=False, zeroline=False, title="X (m)"),
    yaxis=dict(showgrid=False, zeroline=False, title="Y (m)"),
    paper_bgcolor="white", plot_bgcolor="#f5f5f7",
    margin=dict(l=40, r=80, t=80, b=40),
)
fig.show()

In [ ]:
fig2 = go.Figure()

# ── room footprints ────────────────────────────────────────────────────────────
sf_footprints = [fp for sid in sf_room_ids for fp in [get_floor_face(space_by_id[sid]["topology"])] if fp]
for fp in sf_footprints:
    xs, ys = face_outline_xy(fp)
    fig2.add_trace(go.Scatter(
        x=xs, y=ys, fill='toself',
        fillcolor='rgba(200,205,220,0.15)',
        line=dict(color='rgba(110,120,145,0.4)', width=0.4),
        mode='lines', showlegend=False, hoverinfo='skip',
    ))

# ── SF graph edges (thin gray) ─────────────────────────────────────────────────
sf_id_set = set(sf_room_ids)
sf_edge_xs, sf_edge_ys = [], []
for e in Graph.Edges(g):
    everts = Topology.Vertices(e)
    if len(everts) == 2:
        d0 = Topology.Dictionary(everts[0])
        d1 = Topology.Dictionary(everts[1])
        id0 = Dictionary.ValueAtKey(d0, "IFC_id")
        id1 = Dictionary.ValueAtKey(d1, "IFC_id")
        if id0 in sf_id_set and id1 in sf_id_set:
            sf_edge_xs += [everts[0].X(), everts[1].X(), None]
            sf_edge_ys += [everts[0].Y(), everts[1].Y(), None]
fig2.add_trace(go.Scatter(
    x=sf_edge_xs, y=sf_edge_ys, mode='lines',
    line=dict(color='rgba(100,110,140,0.3)', width=1.2),
    showlegend=False, hoverinfo='skip',
))

# ── SF graph nodes (all room centroids) ────────────────────────────────────────
sf_node_xs = [space_by_id[sid]["centroid"].X() for sid in sf_room_ids if sid in vertex_by_ifc_id]
sf_node_ys = [space_by_id[sid]["centroid"].Y() for sid in sf_room_ids if sid in vertex_by_ifc_id]
fig2.add_trace(go.Scatter(
    x=sf_node_xs, y=sf_node_ys, mode='markers',
    marker=dict(color='white', size=7, symbol='circle',
                line=dict(color='rgba(80,90,120,0.6)', width=1.5)),
    showlegend=False, hoverinfo='skip',
))

# ── egress paths (colored lines + nodes) ──────────────────────────────────────
sf_total_lengths = [r["total_length"] for r in sf_results.values() if r["path"] is not None]
sf_mn, sf_mx = min(sf_total_lengths), max(sf_total_lengths)
worst_sf_id = sf_stats[0] if sf_stats else None

for rid, r in sf_results.items():
    if r["path"] is None:
        continue
    xs, ys = wire_xy(r["path"])
    is_worst = rid == worst_sf_id
    color = len_to_color(r["total_length"], sf_mn, sf_mx)
    stair_name = space_by_id[r["stair_sf_id"]]["name"]
    fig2.add_trace(go.Scatter(
        x=xs, y=ys, mode='lines+markers',
        line=dict(color=color, width=6 if is_worst else 3),
        marker=dict(color='white', size=10 if is_worst else 7, symbol='circle',
                    line=dict(color=color, width=2.5 if is_worst else 1.8)),
        showlegend=False,
        hovertemplate=(
            f"{space_by_id[rid]['name']} | {space_by_id[rid]['long_name']}<br>"
            f"→ stairwell {stair_name}<br>"
            f"Leg to stair: {r['dist_to_stair']:.1f} m<br>"
            f"Total combined: {r['total_length']:.1f} m<extra></extra>"
        ),
    ))

fig2.add_trace(go.Scatter(
    x=[None], y=[None], mode='markers',
    marker=dict(
        colorscale=[[0,'rgb(0,200,0)'],[0.5,'rgb(255,200,0)'],[1,'rgb(255,0,0)']],
        cmin=round(sf_mn,1), cmax=round(sf_mx,1), color=[sf_mn, sf_mx],
        colorbar=dict(title=dict(text="Total egress (m)", side="right"),
                      thickness=14, len=0.55, x=1.02, y=0.5, tickfont=dict(size=11)),
        showscale=True,
    ), showlegend=False,
))

for ff_id, sf_id in stair_pairs:
    v = space_by_id[sf_id]["centroid"]
    fig2.add_trace(go.Scatter(
        x=[v.X()], y=[v.Y()], mode='markers+text',
        marker=dict(color='steelblue', size=14, symbol='triangle-up', line=dict(color='navy', width=1.5)),
        text=[space_by_id[sf_id]["name"]], textposition='top center',
        textfont=dict(size=9, color='navy'),
        showlegend=False,
        hovertemplate=f"STAIR: {space_by_id[sf_id]['name']}<extra></extra>",
    ))

if worst_sf_id:
    v = space_by_id[worst_sf_id]["centroid"]
    total = sf_results[worst_sf_id]["total_length"]
    fig2.add_trace(go.Scatter(
        x=[v.X()], y=[v.Y()], mode='markers+text',
        marker=dict(color='darkred', size=18, symbol='star'),
        text=[f"WORST: {space_by_id[worst_sf_id]['name']} ({total:.0f} m total)"],
        textposition='bottom center', textfont=dict(size=11, color='darkred'),
        showlegend=False,
    ))

fig2.update_layout(
    title=dict(
        text="Second Floor — Egress to nearest stairwell (color = total combined distance)<br>"
             "<sup>Green = short · Red = long · Star = worst case · Triangle = stairwell</sup>",
        font=dict(size=15), x=0.5,
    ),
    width=1100, height=750,
    xaxis=dict(scaleanchor="y", scaleratio=1, showgrid=False, zeroline=False, title="X (m)"),
    yaxis=dict(showgrid=False, zeroline=False, title="Y (m)"),
    paper_bgcolor="white", plot_bgcolor="#f5f5f7",
    margin=dict(l=40, r=80, t=80, b=40),
)
fig2.show()

## 10. Limitations & next steps

- **Static travel-distance metric only.** This is a worst-case/average geometric egress distance, not a dynamic simulation — it does not model door widths, congestion, or queueing at exits.
- **Connectivity gaps in the source IFC.** Roughly half the building's spaces are not connected to each other through modeled door connectors alone (48 connected components in the access graph). Some isolation is correct by design (standalone utility rooms with their own exterior door); other cases (e.g. open-plan reception/waiting areas) likely reflect missing door objects in the IFC export rather than a real architectural dead end. Rooms with no modeled path are explicitly listed above rather than silently dropped.
- **Stair travel distance is approximated** (`2x` floor-to-floor elevation) rather than traced through actual `IfcStair` flight/landing geometry.
- **Uniform walking speed / no agent diversity.** As scoped by the original workflow, a more sophisticated analysis would require an agent-based simulation modeling crowding and people with different walking abilities — left as future work.